In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Row
from datetime import datetime
import uuid
import time

job_run_id = str(uuid.uuid4())
job_name = "financial_risk_pipeline"
start_time = time.time()

error_message = None

try:
    # =========================
    # LOAD DATA
    # =========================
    silver_df = spark.read.table("silver_stock_data")

    # =========================
    # DATA QUALITY CHECKS
    # =========================
    null_count = silver_df.filter(col("Close").isNull()).count()
    negative_count = silver_df.filter(col("Close") <= 0).count()

    if null_count > 0 or negative_count > 0:
        raise Exception("Data quality check failed")

    # =========================
    # METRICS
    # =========================
    rows_processed = spark.read.table("gold_stock_risk").count()
    rows_rejected = spark.read.table("silver_invalid_records").count()

    status = "SUCCESS"

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    rows_processed = 0
    rows_rejected = 0

end_time = time.time()
execution_time = end_time - start_time

In [0]:
end_time = time.time()
execution_time = end_time - start_time

In [0]:
%sql
ALTER TABLE ETL_Audit_Log ADD COLUMNS (error_message STRING);

In [0]:
audit_data = [Row(
    job_run_id=job_run_id,
    job_name=job_name,
    job_start_time=datetime.fromtimestamp(start_time),
    job_end_time=datetime.fromtimestamp(end_time),
    execution_time_seconds=float(execution_time),
    rows_processed=int(rows_processed),
    rows_rejected=int(rows_rejected),
    status=status,
    error_message=error_message
)]

In [0]:
null_count = silver_df.filter(col("Close").isNull()).count()

In [0]:
negative_count = silver_df.filter(col("Close") <= 0).count()

In [0]:
duplicate_count = silver_df.count() - silver_df.dropDuplicates(["Date"]).count()

In [0]:
if null_count > 0 or negative_count > 0:
    raise Exception("Data quality check failed")

In [0]:
%sql
SELECT * FROM ETL_Audit_Log ORDER BY job_start_time DESC;

job_run_id,job_name,job_start_time,job_end_time,execution_time_seconds,rows_processed,rows_rejected,status,error_message
411522f1-c9d0-445d-a350-e531e53720c4,financial_risk_pipeline,2026-03-17T12:15:49.142Z,2026-03-17T12:18:23.635Z,154.49311470985413,30,0,SUCCESS,null


In [0]:
%sql
SELECT * FROM gold_stock_risk;

Date,Open,High,Low,Close,Volume,ingestion_timestamp,ingestion_date,processed_timestamp,daily_return,ma_7,ma_30,avg_vol_30,volatility_30,REQUIRES_COMPLIANCE_REVIEW,risk_score
2026-02-02T05:00:00.000Z,259.7869174093499,270.2371306501278,258.9676766447768,269.7575988769531,73913400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,269.7575988769531,269.7575988769531,7.39134E7,null,false,null
2026-02-03T05:00:00.000Z,268.9483513556569,271.6258386480508,267.3598109321867,269.22808837890625,64394700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.001962912259937505,269.4928436279297,269.4928436279297,6.915405E7,0.37442046387841144,false,7.349545189184215E-4
2026-02-04T05:00:00.000Z,272.03545112075,278.68922850453845,272.03545112075,276.23150634765625,90545700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.02601295433518634,271.7390645345052,271.7390645345052,7.62846E7,3.8995666971211462,false,0.10143925041922579
2026-02-05T05:00:00.000Z,277.86999494352693,279.2387093202658,272.97458180817284,275.6520690917969,52977400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.0020976508564164785,272.7173156738281,272.7173156738281,7.04578E7,3.7370641038847494,false,0.007839055717997125
2026-02-06T05:00:00.000Z,276.8609202349588,280.6473855638201,276.67109542368036,277.8599853515625,50453400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.00800979389358529,273.745849609375,273.745849609375,6.645692E7,3.970345875394946,false,0.031801652148159984
2026-02-09T05:00:00.000Z,277.9100036621094,278.20001220703125,271.70001220703125,274.6199951171875,44623400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.011660513946531741,273.89154052734375,273.89154052734375,6.2818E7,3.5690716100526747,false,0.04161720928518971
2026-02-10T05:00:00.000Z,274.8900146484375,275.3699951171875,272.94000244140625,273.67999267578125,34376900,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.0034229206107338488,273.861319405692,273.861319405692,5.875498571428572E7,3.259082694781959,false,0.01115558132805518
2026-02-11T05:00:00.000Z,274.70001220703125,280.17999267578125,274.45001220703125,275.5,51931300,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.006650129249217156,274.6816624232701,274.06615447998047,5.7902025E7,3.0724428367913865,false,0.02043214197549413
2026-02-12T05:00:00.000Z,275.5899963378906,275.7200012207031,260.17999267578125,261.7300109863281,81077200,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.04998181130189428,273.61050851004467,272.69547186957465,6.047704444444445E7,5.016857006971794,false,0.2507516002510503
2026-02-13T05:00:00.000Z,262.010009765625,262.2300109863281,255.4499969482422,255.77999877929688,56290700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.022733396848946215,270.6888645717076,271.00392456054686,6.005841E7,7.140422806562773,false,0.16232606533085783


In [0]:
%sql
SELECT * FROM ETL_Audit_Log;

job_run_id,job_name,job_start_time,job_end_time,execution_time_seconds,rows_processed,rows_rejected,status,error_message
411522f1-c9d0-445d-a350-e531e53720c4,financial_risk_pipeline,2026-03-17T12:15:49.142Z,2026-03-17T12:18:23.635Z,154.49311470985413,30,0,SUCCESS,null
